<a href="https://colab.research.google.com/github/frank-morales2020/MITDevOps/blob/master/AGENTIC_TUTORIAL-ch7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://datasciencedojo.com/bootcamps/agentic-ai-bootcamp/?utm_term=agentic_ai_bootcamp_discount&utm_campaign=Future%20of%20Data%20and%20AI%20Conference&utm_medium=email&_hsenc=p2ANqtz-9FXiLNaw8V3i3MXgDkM_SCN4ThUgM-rGOm2SpfiLw2IbtAzhnj-jshV0z-CYx38T6dYD8tJNoI9624tbqFG-P3pvM3Dw&_hsmi=363333365&utm_content=fodai_agentic_ai_discount&utm_source=email

In [ ]:
!pip install langchain langchain-openai beautifulsoup4 chromadb tiktoken pydantic -q
!pip install langchain_openai -q
!pip install langchain_core -q
!pip install colab-env -q

!pip install langchain_community -q

In [ ]:
import os
import colab_env

# chapter 7: Agentic Workflow Fundamentals

Great! We've covered the foundational concepts of LLM agents, their internal workings, and how to monitor them. Now, let's unlock the true power of autonomous AI with LangGraph's orchestration engine. This is where we build structured, multi-tasking, and collaborative agents that can handle complex, dynamic workflows – perfectly suited for our advanced flight planning AI.

Dive into LangGraph’s Orchestration Engine

Graph-based Orchestration Models:
Traditional AgentExecutor from LangChain is excellent for linear "Thought-Action-Observation" loops. However, real-world problems often require:

* Non-linear flows: Branching based on conditions (e.g., "If flight found, then ask for hotel; else, suggest alternative dates").

* Loops/Cycles: Iterating (e.g., "Keep searching until a suitable flight is found" or "Refine answer until satisfactory").

* Complex Dependencies: Multiple sub-tasks running in parallel or sequentially with intricate data hand-offs.

* Stateful Transitions: The ability for the workflow to "remember" its current progress and make decisions based on accumulating information.

LangGraph solves this by adopting a state machine and graph-based approach. You define nodes (steps) and edges (transitions between steps), allowing for explicit control over the flow, including cycles and conditional routing.

A Practical Guide to Coordinated LLM Agents Using LangGraph:

* Nodes (functions or agents): Each node in a LangGraph represents a specific step in your workflow. A node can be:

A simple Python function (e.g., Notes, format_output).
An LLM call.
A tool invocation.
An entire LangChain AgentExecutor (allowing you to nest agents within a larger graph).

* Edges (data/control flow): These define how the workflow moves from one node to another.
Direct Edges: Unconditionally move from Node A to Node B.
Conditional Edges: The next node is determined by a function that evaluates the current state and returns the name of the next node (or END to stop).

* Cycles (iteration, self-correction): LangGraph natively supports cycles. This is how you implement looping behavior, allowing agents to iteratively refine an answer, gather more information, or retry failed operations.

* State: LangGraph manages a shared state object that is passed between nodes. Each node receives the current state, performs its operations, and returns an update to the state. This allows all nodes to access and contribute to the accumulated knowledge and progress of the workflow. The state is defined using a TypedDict or a simple dict that represents the data structure of your graph's context.

Add memory or context passing between agents:
In LangGraph, memory is intrinsically tied to the state. The state object itself serves as the short-term memory of the current execution trace. You can design your state to include a messages list for conversational memory, or specific keys for extracted entities and results. LangGraph also provides Checkpointers for persisting this state to a database, enabling long-running conversations or resuming workflows after interruptions.

Node-based Task Design:
Think of each node as a microservice or a single, atomic operation.

* Atomic: Each node should ideally perform one distinct logical unit of work.

* Input/Output: Clearly define what each node expects from the state and what it contributes back to the state.

* Separation of Concerns: Keep your planner, executor, summarizer, and specific tool interactions in separate nodes for clarity and maintainability.


Async vs Sync Execution in Agentic Flows:

LangGraph (and LangChain) supports both synchronous (.invoke()) and asynchronous (.ainvoke(), .stream(), .astream()) execution.

* Synchronous: Simpler to write for sequential tasks, blocking until each step completes.

* Asynchronous: Essential for high-throughput, non-blocking operations, especially when dealing with network calls (LLMs, external APIs) or parallel execution. LangGraph can internally manage async operations within nodes if you define your nodes as async def functions.

Conditional Routing and Stateful Transitions:

This is a cornerstone of LangGraph. You define a "router" function as an edge that inspects the current state and returns the name of the next node to execute.

Integrating memory into LangGraph Workflows:

* Conversation History: A common pattern is to have a messages key in your graph state, and use LangGraph's add_messages function (or a custom reducer) to append new messages to it.

* Structured Data: Store extracted entities, API results, or summaries directly in dedicated keys within your state.

* Checkpointers: For persistence, you compile the graph with a checkpointer (e.g., InMemorySaver for development, or a database-backed one for production). This allows you to resume conversations or long-running tasks.


Hands-on Exercises Using LangGraph
We will build a complete trip planning system using LangGraph. This system will:

1. Take a user request.
2. Use a Planner Node to break it into tasks.
3. Use an Executor Node that conditionally calls FlightSearchAgent or HotelBookingAgent.
4. Implement looping behavior to process multiple tasks.
5. Use a Summarizer Node to compile results.
6. Maintain state throughout the process.
7. Include an Error Handling Node for robustness.

We'll reuse our FlightSearchAgent and HotelBookingAgent from the previous section as internal components (called by the Executor).

Prerequisites:

In [ ]:
!pip install langchain langchain-openai langgraph -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.9/154.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 15.4 MB/s eta 0:00:00


In [ ]:
#!pip install langchain langchain-openai langgraph -q
import os
import json
import time
from typing import List, Dict, Any, TypedDict, Callable, Optional
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langgraph.graph import StateGraph, END, START
from langgraph.graph.graph import CompiledGraph
from langgraph.checkpoint.memory import InMemorySaver # For state persistence
from langchain_core.output_parsers import StrOutputParser # Import StrOutputParser

# --- Re-initialize LLM (as in previous steps) ---
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- Re-define Specialized Agents (Flight and Hotel) ---
# These are still regular LangChain Agents, which will be invoked as "tools"
# within our LangGraph nodes.

class FlightSearchAgent:
    def __init__(self, llm_instance):
        self.llm = llm_instance
        @tool("search_flights_api", args_schema={"origin": str, "destination": str, "date": str})
        def _search_flights_tool(origin: str, destination: str, date: str) -> List[dict]:
            """Searches for flights between origin and destination on a specific date.""" # Added docstring
            print(f"  [FlightSearchAgent] Simulating Flight API call for: {origin} -> {destination} on {date}")
            time.sleep(0.5)
            if "Montreal" in origin and "Paris" in destination and "2025-08-15" in date:
                return [{"flight_id": "AC123", "price": 750, "airline": "Air Canada"}, {"flight_id": "AF456", "price": 820, "airline": "Air France"}]
            elif "Paris" in origin and "Rome" in destination and "2025-08-20" in date:
                return [{"flight_id": "AZ789", "price": 120, "airline": "Alitalia"}, {"flight_id": "AF101", "price": 150, "airline": "Air France"}]
            return []
        self.tools = [_search_flights_tool]
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful flight search assistant. Use the 'search_flights_api' tool."),
            ("user", "{input}"), ("placeholder", "{agent_scratchpad}")])
        self.agent = create_tool_calling_agent(self.llm, self.tools, self.prompt)
        self.executor = AgentExecutor(agent=self.agent, tools=self.tools, verbose=False)

    def run(self, query: str) -> str:
        try:
            result = self.executor.invoke({"input": query})
            return result['output']
        except Exception as e:
            return f"ERROR: Flight Search Agent failed with {str(e)}"

class HotelBookingAgent:
    def __init__(self, llm_instance):
        self.llm = llm_instance
        @tool("Google Hotels_api", args_schema={"location": str, "check_in_date": str, "check_out_date": str, "num_guests": int})
        def _Google_Hotels_tool(location: str, check_in_date: str, check_out_date: str, num_guests: int) -> List[dict]:
            """Searches for hotels in a given location for specified dates and number of guests.""" # Added docstring
            print(f"  [HotelBookingAgent] Simulating Hotel API call for: {location} for {num_guests} guests")
            time.sleep(0.3)
            if "Paris" in location and "2025-08-15" in check_in_date:
                return [{"hotel_id": "HP789", "name": "Hotel Paradis", "price_per_night": 200, "stars": 4}, {"hotel_id": "RS012", "name": "Riverside Suites", "price_per_night": 150, "stars": 3}]
            elif "Rome" in location and "2025-08-20" in check_in_date:
                return [{"hotel_id": "GHZXC", "name": "Grand Hotel Roma", "price_per_night": 180, "stars": 4}]
            return []
        self.tools = [_Google_Hotels_tool]
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful hotel search assistant. Use the 'Google Hotels_api' tool."),
            ("user", "{input}"), ("placeholder", "{agent_scratchpad}")])
        self.agent = create_tool_calling_agent(self.llm, self.tools, self.prompt)
        self.executor = AgentExecutor(agent=self.agent, tools=self.tools, verbose=False)

    def run(self, query: str) -> str:
        try:
            result = self.executor.invoke({"input": query})
            return result['output']
        except Exception as e:
            return f"ERROR: Hotel Search Agent failed with {str(e)}"

# Instantiate our specialized "service" agents (these will be called by the Executor node)
flight_service = FlightSearchAgent(llm)
hotel_service = HotelBookingAgent(llm)

# --- Define the Graph State ---
# This TypedDict defines the structure of the data that will be passed between nodes.
class TravelGraphState(TypedDict):
    request: str # The original user request
    tasks: List[Dict[str, Any]] # List of tasks generated by the Planner
    current_task_index: int # Index of the task being processed
    flight_results: List[Dict[str, Any]] # Accumulated flight results
    hotel_results: List[Dict[str, Any]] # Accumulated hotel results
    error_message: Optional[str] # For error handling
    final_itinerary: Optional[str] # The final output

# --- Define the Nodes ---

# Node 1: Planner Node
def planner_node(state: TravelGraphState) -> TravelGraphState:
    """
    Takes the user request and breaks it down into a list of tasks.
    Updates the 'tasks' and 'current_task_index' in the state.
    """
    print("\n[Node: Planner] Breaking down the request...")
    # Escape curly braces that are part of the JSON format description
    # CORRECTED: Escaped curly braces in the system message example JSON
    planner_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a meticulous task planner for a travel agent.
         Your job is to break down a user's travel request into a precise sequence of discrete tasks.
         Each task must be a JSON object with:
         - 'task': (string) A concise description of the sub-task.
         - 'type': (string) The type of task, e.g., 'flight_search', 'hotel_search'.
         - 'details': (dict) Key-value pairs of parameters for the task.
           For 'flight_search': {{"origin"}}, {{"destination"}}, {{"date"}}.
           For 'hotel_search': {{"location"}}, {{"check_in_date"}}, {{"check_out_date"}}, {{"num_guests"}}.
         Ensure you extract all necessary details for each task.
         Return only a JSON list of task objects.
         Example: `[{{\"task\": \"Find flights from NYC to London\", \"type\": \"flight_search\", \"details\": {{\"origin\": \"NYC\", \"destination\": \"London\", \"date\": \"2025-07-01\"}}}}, {{\\"task\\": \\"Find hotel in London\\", \\"type\\": \\"hotel_search\\", \\"details\\": {\\"location\\": \\"London\\", \\"check_in_date\\": \\"2025-07-01\\", \\"check_out_date\\": \\"2025-07-05\\", \\"num_guests\\": 1}}}]`
         """),
        ("human", "User Request: {request}. Assume today's date is 2025-05-26 if relative dates are used."),
    ])
    try:
        # Ensure the output parser correctly handles potential non-JSON responses from the LLM
        # and provides a helpful error. We'll keep the lambda x: json.loads(x) for now,
        # but add a more specific error message around it.
        planner_chain = planner_prompt | llm | (lambda x: json.loads(x))
        tasks_raw = planner_chain.invoke({"request": state['request']})

        # Basic validation - check if the top level is a list
        if not isinstance(tasks_raw, list):
             raise ValueError(f"Planner did not return a JSON list. Raw output: {tasks_raw}")

        tasks = []
        for t in tasks_raw:
            # Basic validation - check if each item is a dict and has required keys
            # CORRECTED: Also check if 'details' is a dictionary
            if isinstance(t, dict) and all(k in t for k in ['task', 'type', 'details']) and isinstance(t.get('details'), dict): # Use .get for safer access
                tasks.append(t)
            else:
                # Catch malformed *items* in the list, provide better error
                raise ValueError(f"Invalid task format received from planner: {t}")


        print(f"[Node: Planner] Identified {len(tasks)} tasks.")
        # Ensure current_task_index is always returned on success
        return {"tasks": tasks, "current_task_index": 0}
    except json.JSONDecodeError as e:
        error_msg = f"Planner output was not valid JSON. Error: {e}. Raw output: {tasks_raw if 'tasks_raw' in locals() else 'N/A'}"
        print(f"[Node: Planner] Error planning tasks: {error_msg}")
        # Ensure state keys are present even on error for graph consistency
        return {"error_message": error_msg, "tasks": [], "current_task_index": 0}
    except Exception as e:
        error_msg = f"Error planning tasks: {e}"
        print(f"[Node: Planner] Error planning tasks: {error_msg}")
        # Ensure state keys are present even on error for graph consistency
        return {"error_message": error_msg, "tasks": [], "current_task_index": 0}


# Node 2: Executor Node
def executor_node(state: TravelGraphState) -> TravelGraphState:
    """
    Executes the current task using the appropriate specialized agent/tool.
    Updates 'flight_results' or 'hotel_results' in the state.
    Advances 'current_task_index'.
    """
    # Check for error message from previous node before proceeding
    if state.get("error_message"):
        print("[Node: Executor] Skipping execution due to previous error.")
        return state # Return current state, error_handler will be next via router

    # Basic check for expected keys
    if 'tasks' not in state or 'current_task_index' not in state:
         error = "Internal workflow error: missing task information in executor."
         print(f"[Node: Executor] {error}")
         return {"error_message": error} # This state update will be caught by the router

    print(f"\n[Node: Executor] Executing task index: {state['current_task_index']}")
    tasks = state['tasks']
    current_index = state['current_task_index']

    if current_index >= len(tasks):
        print("[Node: Executor] No more tasks to execute.")
        return state # Should not happen if conditional routing is correct

    task = tasks[current_index]
    # Use .get for safety when accessing task keys
    task_type = task.get('type')
    task_details = task.get('details', {}) # Default to empty dict if details are missing

    new_flight_results = state.get('flight_results', [])
    new_hotel_results = state.get('hotel_results', [])
    error = None # Initialize error to None for this node's execution

    try:
        if task_type == 'flight_search':
             # Use .get for safety when accessing task_details keys
            origin = task_details.get('origin')
            destination = task_details.get('destination')
            date = task_details.get('date')
            if not all([origin, destination, date]):
                 raise ValueError(f"Missing required details for flight search: {task_details}")
            query = f"Find flights from {origin} to {destination} on {date}."
            result_str = flight_service.run(query) # Call the specialized agent
            if result_str and "ERROR" in result_str: raise Exception(result_str)
            # Attempt to parse result, handle potential non-JSON
            try:
                flights = json.loads(result_str)
                if not isinstance(flights, list):
                     raise ValueError(f"Flight service did not return a list: {result_str}")
                new_flight_results.extend(flights)
                print(f"[Node: Executor] Found {len(flights)} flights.")
            except json.JSONDecodeError:
                 raise ValueError(f"Flight service returned non-JSON data: {result_str}")


        elif task_type == 'hotel_search':
            # Use .get for safety when accessing task_details keys
            location = task_details.get('location')
            check_in_date = task_details.get('check_in_date')
            check_out_date = task_details.get('check_out_date')
            num_guests = task_details.get('num_guests')
            if not all([location, check_in_date, check_out_date, num_guests]):
                 raise ValueError(f"Missing required details for hotel search: {task_details}")
            # Ensure num_guests is an integer if it came from a string
            try:
                num_guests = int(num_guests) if isinstance(num_guests, str) else num_guests
                if not isinstance(num_guests, int):
                     raise ValueError(f"Invalid num_guests format: {num_guests}")
            except ValueError:
                 raise ValueError(f"Could not convert num_guests to integer: {num_guests}")


            query = f"Find hotels in {location} from {check_in_date} to {check_out_date} for {num_guests} guests."
            result_str = hotel_service.run(query) # Call the specialized agent
            if result_str and "ERROR" in result_str: raise Exception(result_str)
            # Attempt to parse result, handle potential non-JSON
            try:
                hotels = json.loads(result_str)
                if not isinstance(hotels, list):
                     raise ValueError(f"Hotel service did not return a list: {result_str}")
                new_hotel_results.extend(hotels)
                print(f"[Node: Executor] Found {len(hotels)} hotels.")
            except json.JSONDecodeError:
                 raise ValueError(f"Hotel service returned non-JSON data: {result_str}")

        elif task_type is None:
             raise ValueError(f"Task dictionary missing 'type' key: {task}")
        else:
            error = f"Unsupported task type: {task_type}"
            print(f"[Node: Executor] {error}")
            # For unsupported tasks, maybe skip and move to next? Or treat as error?
            # Let's treat as error for now for robustness.

    except Exception as e:
        error = f"Error executing task '{task.get('task', 'Unknown Task')}': {e}"
        print(f"[Node: Executor] {error}")

    next_index = current_index + 1

    # Update state
    return {
        "flight_results": new_flight_results,
        "hotel_results": new_hotel_results,
        "current_task_index": next_index,
        # Preserve existing error_message if any, only overwrite if a *new* error occurred in *this* node
        "error_message": state.get("error_message") or error
    }


# Node 3: Summarizer Node
def summarizer_node(state: TravelGraphState) -> TravelGraphState:
    """
    Compiles all collected results into a final itinerary.
    Updates 'final_itinerary' in the state.
    """
    # Check for error message from previous nodes before proceeding
    if state.get("error_message"):
        print("[Node: Summarizer] Skipping summarization due to previous error.")
        return state # Return current state, error_handler will be next via router

    print("\n[Node: Summarizer] Compiling final itinerary...")
    summarizer_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a professional travel itinerary creator.
         You receive raw flight and hotel search results along with the original user request.
         Compile all the information into a clear, attractive, and concise travel itinerary.
         Highlight key details: dates, locations, flight numbers, airlines, departure/arrival times, hotel names, prices, and number of guests.
         If any information is missing or could not be found, clearly state it.
         Start with a friendly greeting and provide a well-structured summary.
         """),
        ("human", "Original Request: {original_request}\n\nSearch Results:\nFlights: {flights}\nHotes: {hotels}"),
    ])

    try:
        summary_chain = summarizer_prompt | llm | StrOutputParser()
        final_itinerary = summary_chain.invoke({
            "original_request": state.get('request', 'User request unavailable.'), # Use .get for safety
            "flights": state.get('flight_results', []),
            "hotels": state.get('hotel_results', [])
        })
        print("[Node: Summarizer] Itinerary compiled.")
        # Return final_itinerary and ensure error_message is cleared if summarization was successful
        return {"final_itinerary": final_itinerary, "error_message": None}
    except Exception as e:
        print(f"[Node: Summarizer] Error summarizing itinerary: {e}")
        # Ensure error_message is returned to state
        return {"error_message": f"Error summarizing: {e}"}

# Node 4: Error Handler Node (Optional but good for robustness)
def error_handler_node(state: TravelGraphState) -> TravelGraphState:
    """
    Handles errors that occurred in previous nodes.
    Formats a user-friendly error message for the final output.
    """
    print(f"\n[Node: Error Handler] Processing error: {state.get('error_message', 'An unknown error occurred.')}")
    error_summary = f"I encountered an issue during your request: {state.get('error_message', 'An unknown error occurred.')}. Please try again or rephrase your request."
    # Return the final itinerary as the error message
    return {"final_itinerary": error_summary, "error_message": state.get('error_message')}


# --- Define the Conditional Edges (Routers) ---

def should_continue_execution(state: TravelGraphState) -> str:
    """
    Decides whether to continue executing tasks or move to summarization/error.
    This router is used AFTER the executor node.
    """
    # Check for error message from the current node execution first
    if state.get("error_message"):
        print("[Router (after Executor)] Routing to Error Handler due to error.")
        return "error_handler"

    # Basic check for expected keys - shouldn't be missing if Planner ran successfully and
    # Executor returned the required keys, but defensive check is good.
    if 'tasks' not in state or 'current_task_index' not in state:
         print("[Router (after Executor)] Missing 'tasks' or 'current_task_index' in state after executor. Routing to error.")
         # Note: The executor node should ideally set the error_message if these were missing,
         # but we add a fallback here.
         # state["error_message"] = "Internal workflow error: missing task information after executor."
         return "error_handler"


    current_index = state['current_task_index']
    tasks = state['tasks']

    if current_index < len(tasks):
        # More tasks to execute, go back to the executor
        print(f"[Router (after Executor)] More tasks to execute. Routing to Executor (Task {current_index + 1}/{len(tasks)}).")
        return "executor"
    else:
        # All tasks processed, go to the summarizer
        print("[Router (after Executor)] All tasks processed. Routing to Summarizer.")
        return "summarizer"


# --- Build the LangGraph Workflow ---

builder = StateGraph(TravelGraphState)

# Add nodes
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("summarizer", summarizer_node)
builder.add_node("error_handler", error_handler_node) # Add error handler node

# Set entry point
builder.set_entry_point("planner")

# Add edges
# Conditional routing from planner: check for error message immediately after planner
builder.add_conditional_edges(
    "planner",
    # Lambda function checks for 'error_message' key in state
    lambda state: "error_handler" if state.get("error_message") else "executor",
    {
        "executor": "executor",
        "error_handler": "error_handler"
    }
)


# Conditional routing from executor: loop back to executor or go to summarizer/error
# This router checks the state *after* the executor node completes its execution.
builder.add_conditional_edges(
    "executor",
    should_continue_execution, # This function determines next node
    {
        "executor": "executor",       # Loop back to executor if more tasks
        "summarizer": "summarizer",   # Go to summarizer if all tasks done
        "error_handler": "error_handler" # Go to error handler if error occurred (either from executor or earlier)
    }
)

# After summarizer or error, the graph ends
builder.add_edge("summarizer", END)
builder.add_edge("error_handler", END) # Error handler now goes to END

# Compile the graph
# Use checkpointer for state persistence, allowing you to resume runs.
# For simplicity, InMemorySaver is used. For production, use Redis, SQLite, etc.
workflow = builder.compile(checkpointer=InMemorySaver())

# Optional: Visualize the graph (requires graphviz and pydot)
# try:
#     from IPython.display import Image, display
#     display(Image(workflow.get_graph().draw_mermaid_png()))
# except Exception:
#     print("Could not display graph. Ensure graphviz and pydot are installed.")


# --- Hands-on Exercises Using LangGraph ---

print("--- LangGraph Multi-Agent Trip Planning Workflow ---")

# Use a checkpointer for state persistence, allowing you to resume runs.
# For simplicity, InMemorySaver is used. For production, use Redis, SQLite, etc.
# The workflow was already compiled with the checkpointer above, no need to re-compile here.

# --- Exercise 1: Multi-city trip with flights and hotels ---
user_request_1 = "Plan a trip for me: I need flights from Montreal to Paris on August 15, 2025, a hotel in Paris for August 15-20, 2025 for 2 guests, then flights from Paris to Rome on August 20, 2025, and finally a hotel in Rome from August 20-22, 2025 for 2 guests."
print(f"\nUser Request: {user_request_1}")
# `invoke` will run the entire graph
# FIX: Add config with a unique thread_id for the checkpointer
result_1 = workflow.invoke(
    {"request": user_request_1},
    config={"configurable": {"thread_id": "trip_planning_1"}} # Use a unique ID for this run
)
print("\n--- FINAL ITINERARY ---")
print(result_1['final_itinerary'])
print("-----------------------\n")

# --- Exercise 2: Single task request ---
user_request_2 = "Find flights from Montreal to Paris on August 15, 2025."
print(f"\nUser Request: {user_request_2}")
# FIX: Add config with a unique thread_id for the checkpointer
result_2 = workflow.invoke(
    {"request": user_request_2},
    config={"configurable": {"thread_id": "trip_planning_2"}} # Use another unique ID
)
print("\n--- FINAL ITINERARY ---")
print(result_2['final_itinerary'])
print("-----------------------\n")

# --- Exercise 3: Request that might trigger an error (e.g., unparseable date or location for a service) ---
# Modify a specialized agent to throw an error for certain inputs to test.
# For this demo, let's make an input that planner might struggle with or executor gets no data from.
# Or you can add a deliberate error in one of the service.run methods for testing.
# Let's simulate an executor error if "unknown_city" is in the request.
class FaultyHotelBookingAgent(HotelBookingAgent):
    def run(self, query: str) -> str:
        if "unknown_city" in query.lower():
            print("  [FaultyHotelBookingAgent] Simulating an API error for unknown_city.")
            # Return a string that indicates an error, matching the check in executor_node
            return "ERROR: API returned an error for unknown city data."
        # Call the original run method for other inputs
        return super().run(query)

# Temporarily replace the hotel_service with the faulty one for testing purposes
temp_hotel_service = hotel_service
hotel_service = FaultyHotelBookingAgent(llm) # Inject the faulty agent

user_request_3 = "I need a hotel in unknown_city from 2025-08-15 to 2025-08-20 for 1 guest."
print(f"\nUser Request: {user_request_3}")
# FIX: Add config with a unique thread_id for the checkpointer
result_3 = workflow.invoke(
    {"request": user_request_3},
    config={"configurable": {"thread_id": "trip_planning_3"}} # Use yet another unique ID
)
print("\n--- FINAL ITINERARY ---")
print(result_3['final_itinerary'])
print("-----------------------\n")

# Revert to original hotel service
hotel_service = temp_hotel_service

--- LangGraph Multi-Agent Trip Planning Workflow ---

User Request: Plan a trip for me: I need flights from Montreal to Paris on August 15, 2025, a hotel in Paris for August 15-20, 2025 for 2 guests, then flights from Paris to Rome on August 20, 2025, and finally a hotel in Rome from August 20-22, 2025 for 2 guests.

[Node: Planner] Breaking down the request...
[Node: Planner] Error planning tasks: Error planning tasks: 'Input to ChatPromptTemplate is missing variables {\'\\\\"location\\\\"\'}.  Expected: [\'\\\\"location\\\\"\', \'request\'] Received: [\'request\']\nNote: if you intended {\\"location\\"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{\\"location\\"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

[Node: Error Handler] Processing error: Error planning tasks: 'Input to ChatPromptTemplate is missing variables {\'\\\\"location\\\\"\'}.  Expected: [\'\\\\"loc

Explanation and Key Learnings:

1. Graph State (TravelGraphState): This is the central piece of LangGraph. It defines the schema of the information that is passed between nodes. Each node receives the current state, updates it, and returns the modified state.

2. Nodes (Functions):
planner_node: Takes the initial request, uses an LLM to parse it into structured tasks, and updates state['tasks'] and state['current_task_index'].
executor_node: This is the workhorse. It reads the current_task_index from the state, executes the corresponding task by calling our specialized flight_service.run() or hotel_service.run() (simulating API calls to other agents), and then appends the results to state['flight_results'] or state['hotel_results']. It increments current_task_index.
summarizer_node: Takes all accumulated flight_results and hotel_results from the state and uses another LLM call to synthesize the final_itinerary.
error_handler_node: A dedicated node to provide a graceful response if an error occurred during processing.

3. Conditional Edges (should_continue_execution): This function is the "router" that defines the graph's flow.

* * It's called after the executor node.
* * It inspects the state (specifically current_task_index and error_message).
* * If error_message is present, it routes to error_handler.
* * If there are more tasks to execute (current_task_index < len(tasks)), it routes back to the executor node, creating a loop.
* * Otherwise (all tasks processed), it routes to the summarizer node.

4. Cycles (Looping Behavior): The edge executor -> should_continue_execution -> executor forms a cycle, enabling the graph to iteratively process each task identified by the planner.

5. Stateful Transitions: The state is continuously updated by each node. This allows subsequent nodes (and even subsequent iterations of the same node) to access the results and context from previous steps. For example, the summarizer_node receives all results accumulated by multiple runs of the executor_node.

6. Integrating Memory: The TravelGraphState itself serves as the active memory for the current workflow execution. For persistence across sessions, workflow.compile() accepts a checkpointer (e.g., InMemorySaver for local, or more robust options like SqliteSaver or Redis for production). This allows you to invoke with a thread_id and resume a previous conversation.

7. Node-based Task Design: Each node has a clear, singular responsibility (planning, executing one task, summarizing, handling errors), making the system modular and easier to debug.

LangGraph provides a powerful, explicit, and visual way to design complex agentic workflows, moving beyond simple linear chains to truly dynamic and robust multi-agent systems for challenging tasks like flight planning.